# MediCare Patient Follow-Up Agent

## Setup & Imports

In [3]:
# Install dependencies
!pip install langchain langchain-openai pandas tabulate -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(
    model="openrouter/owl-alpha",
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
)
llm.invoke([HumanMessage(content="Hello, how are you?")])

AIMessage(content="Hello! I'm doing well, thank you for asking. How can I help you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 16, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'openrouter/owl-alpha', 'system_fingerprint': None, 'id': 'gen-1782672286-GUX56pF7X0SG2b2GiuxN', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f0f8c-612d-7fd1-82f1-97a92883e8af-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 20, 'total_t

In [4]:
import os
import pandas as pd
import json
from datetime import datetime

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.prebuilt import create_react_agent

# ── LLM ───────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model="openrouter/owl-alpha",
    api_key="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",
    base_url="https://openrouter.ai/api/v1",
)
print("✅ LLM initialised:", llm.model)

✅ LLM initialised: openrouter/owl-alpha


In [5]:
response = llm.invoke([
    HumanMessage(content="How many r's are in the word 'strawberry'?")
])

print(response.content)

Let me count the r's in 'strawberry':

s-t-**r**-a-w-b-e-**r**-**r**-y

There are **3** r's in 'strawberry'.


## Task 1 — Data Loading & Exploratory Analysis

In [6]:
df = pd.read_csv("patient_data.csv", parse_dates=["last_visit_date", "next_scheduled_visit"])
print(f"Dataset: {df.shape[0]} patients × {df.shape[1]} columns")
df.head(3)

Dataset: 100 patients × 18 columns


,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,next_scheduled_visit,days_since_last_visit,missed_last_appointment,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2,notes
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,2024-06-08,251,No,174,70,95,96,Patient reports good medication adherence.
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,2024-02-28,352,No,174,91,72,97,Patient reports no new symptoms.
2,P0003,Rohan Mehta,45,Male,"Type 2 Diabetes, CKD","Insulin Glargine 10U, Losartan 50mg",HbA1c,11.4,%,2024-08-04,2024-10-03,149,No,132,86,64,91,Medication dose adjusted this visit.


In [7]:
# Quick EDA
print("=== Diagnosis Distribution ===")
print(df["diagnosis"].value_counts().head(10).to_string())

print("\n=== Missed Appointments ===")
print(df["missed_last_appointment"].value_counts().to_string())

print("\n=== Key Vitals Summary ===")
print(df[["vitals_bp_systolic", "vitals_bp_diastolic", "vitals_heart_rate", "vitals_spo2"]].describe().round(1).to_string())

print("\n=== Days Since Last Visit ===")
print(df["days_since_last_visit"].describe().round(1).to_string())

=== Diagnosis Distribution ===
diagnosis
Heart Failure                    12
Hypothyroidism                    9
Hypertension, Anemia              8
COPD                              7
Osteoarthritis                    7
Type 2 Diabetes, Hypertension     7
Depression                        7
Type 2 Diabetes                   6
COPD, Heart Failure               6
Asthma                            6

=== Missed Appointments ===
missed_last_appointment
No     85
Yes    15

=== Key Vitals Summary ===
       vitals_bp_systolic  vitals_bp_diastolic  vitals_heart_rate  vitals_spo2
count               100.0                100.0              100.0        100.0
mean                145.6                 87.7               82.4         94.5
std                  20.4                 13.6               14.5          2.8
min                 105.0                 65.0               58.0         90.0
25%                 130.0                 75.8               72.0         92.0
50%                 145.

In [8]:
print("=" * 55)
print("TASK 1 — EXPLORATORY DATA ANALYSIS")
print("=" * 55)

print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDiagnosis Distribution:\n{df['diagnosis'].value_counts().head(10)}")
print(f"\nMissed Appointments:\n{df['missed_last_appointment'].value_counts().to_dict()}")
print(f"\nAvg Days Since Last Visit: {df['days_since_last_visit'].mean():.1f}")
print(f"\nOverdue > 180 days: {(df['days_since_last_visit'] > 180).sum()} patients")
print(f"\nLab Test Types:\n{df['lab_test'].value_counts()}")
print(f"\nAge Stats:\n{df['age'].describe()}")
print(f"\nNull Values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

TASK 1 — EXPLORATORY DATA ANALYSIS

Shape: (100, 18)

Columns: ['patient_id', 'patient_name', 'age', 'gender', 'diagnosis', 'current_medication', 'lab_test', 'lab_value', 'lab_unit', 'last_visit_date', 'next_scheduled_visit', 'days_since_last_visit', 'missed_last_appointment', 'vitals_bp_systolic', 'vitals_bp_diastolic', 'vitals_heart_rate', 'vitals_spo2', 'notes']

Diagnosis Distribution:
diagnosis
Heart Failure                    12
Hypothyroidism                    9
Hypertension, Anemia              8
COPD                              7
Osteoarthritis                    7
Type 2 Diabetes, Hypertension     7
Depression                        7
Type 2 Diabetes                   6
COPD, Heart Failure               6
Asthma                            6
Name: count, dtype: int64

Missed Appointments:
{'No': 85, 'Yes': 15}

Avg Days Since Last Visit: 211.4

Overdue > 180 days: 57 patients

Lab Test Types:
lab_test
BP Systolic    18
HbA1c          17
FEV1%          13
BNP            12
TS

## Task 2 — Define Agent Tools

In [9]:
# Clinical thresholds reference
THRESHOLDS = {
    "HbA1c":      {"high": 7.0,   "unit": "%",    "condition": "Type 2 Diabetes"},
    "BNP":        {"high": 100,   "unit": "pg/mL","condition": "Heart Failure"},
    "Creatinine": {"high": 1.2,   "unit": "mg/dL","condition": "CKD"},
    "Hemoglobin": {"low":  12.0,  "unit": "g/dL", "condition": "Anemia"},
    "FEV1":       {"low":  70.0,  "unit": "%",    "condition": "COPD"},
}

@tool
def get_patient_record(patient_id: str) -> str:
    """Retrieve full clinical record for a single patient by patient_id (e.g. 'P0001')."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return f"No patient found with ID {patient_id}."
    return row.to_dict(orient="records")[0].__str__()


@tool
def flag_lab_risk(patient_id: str) -> str:
    """Check whether a patient's latest lab value exceeds clinical thresholds. Returns risk level and explanation."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return "Patient not found."
    r = row.iloc[0]
    lab, val = r["lab_test"], float(r["lab_value"])
    thresh = THRESHOLDS.get(lab)
    if not thresh:
        return f"No threshold defined for lab test '{lab}'."
    if "high" in thresh and val > thresh["high"]:
        return (f"HIGH RISK — {lab} = {val} {thresh['unit']} "
                f"(threshold > {thresh['high']}). Condition: {thresh['condition']}.")
    if "low" in thresh and val < thresh["low"]:
        return (f"HIGH RISK — {lab} = {val} {thresh['unit']} "
                f"(threshold < {thresh['low']}). Condition: {thresh['condition']}.")
    return f"NORMAL — {lab} = {val} {thresh['unit']} within acceptable range."


@tool
def flag_vitals_risk(patient_id: str) -> str:
    """Assess whether a patient's blood pressure, heart rate, or SpO2 are outside safe ranges."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return "Patient not found."
    r = row.iloc[0]
    flags = []
    if r["vitals_bp_systolic"] >= 140 or r["vitals_bp_diastolic"] >= 90:
        flags.append(f"Hypertension: BP {r['vitals_bp_systolic']}/{r['vitals_bp_diastolic']} mmHg")
    if r["vitals_heart_rate"] > 100:
        flags.append(f"Tachycardia: HR {r['vitals_heart_rate']} bpm")
    if r["vitals_spo2"] < 92:
        flags.append(f"Hypoxia: SpO2 {r['vitals_spo2']}%")
    return "VITALS RISKS: " + "; ".join(flags) if flags else "Vitals within normal limits."


@tool
def get_missed_appointment_patients(top_n: int = 10) -> str:
    """Return patients who missed their last appointment, sorted by days since last visit (most overdue first)."""
    missed = (df[df["missed_last_appointment"].str.lower() == "yes"]
              .sort_values("days_since_last_visit", ascending=False)
              .head(top_n)[["patient_id", "patient_name", "diagnosis",
                            "days_since_last_visit", "next_scheduled_visit"]])
    if missed.empty:
        return "No missed appointments found."
    return missed.to_string(index=False)


@tool
def generate_care_action_plan(patient_id: str, risk_summary: str) -> str:
    """Given a patient_id and a plain-text risk_summary, produce a structured follow-up care action plan."""
    row = df[df["patient_id"] == patient_id]
    if row.empty:
        return "Patient not found."
    r = row.iloc[0]
    plan = {
        "patient_id": patient_id,
        "patient_name": r["patient_name"],
        "diagnosis": r["diagnosis"],
        "risk_summary": risk_summary,
        "recommended_actions": "[LLM will fill this in]",
        "priority": "[LLM will fill this in]",
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M"),
    }
    return json.dumps(plan, indent=2)


tools = [get_patient_record, flag_lab_risk, flag_vitals_risk,
         get_missed_appointment_patients, generate_care_action_plan]
print(f"✅ {len(tools)} tools registered:", [t.name for t in tools])

✅ 5 tools registered: ['get_patient_record', 'flag_lab_risk', 'flag_vitals_risk', 'get_missed_appointment_patients', 'generate_care_action_plan']


## Task 3 — Implement the Agentic Loop

In [10]:
SYSTEM_PROMPT = """You are MediCare's AI clinical follow-up agent.
Your job is to autonomously analyse patient data, identify clinical risks,
and generate prioritised care actions for the care coordination team.

Use the available tools to:
1. Summarise the patient population
2. Identify the top high-risk patients
3. For each high-risk patient, generate a clear, actionable follow-up plan

Be concise, clinical, and prioritise patient safety.
Always ground your decisions in tool outputs — do not guess or hallucinate values."""

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

def run_agent(query: str) -> str:
    """Run the agent and return the final text response."""
    result = agent_executor.invoke({
        "messages": [("human", query)]
    })
    return result["messages"][-1].content

print("✅ Agent initialised successfully")

✅ Agent initialised successfully


/tmp/ipykernel_2739/3631975086.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


## Task 4 — Single Patient Analysis

In [17]:
PATIENT_ID = "P0001"   # Change to any patient_id from the CSV

result = agent_executor.invoke({
    "input": f"Perform a full clinical risk assessment for patient {PATIENT_ID} "
             f"and generate a prioritised follow-up care action plan."
})
print("\n" + "="*60)
print("AGENT OUTPUT")
print("="*60)
print(result["messages"][-1].content)


AGENT OUTPUT
---

# 🏥 MediCare Clinical Follow-Up Report

## Population Summary
- **10 patients** identified as having missed their last appointment, ranging from **221–340 days overdue**
- All patients have at least one chronic condition (diabetes, hypertension, CKD, asthma, anemia, hypothyroidism, or osteoarthritis)
- **4 patients flagged as highest priority** based on combined vitals and lab risk

---

## 🔴 CRITICAL PRIORITY

### 1. Paramita Chatterjee (P0099) — 70yo Male
| Factor | Value | Risk |
|---|---|---|
| Days overdue | **340** | Most overdue |
| HbA1c | **9.8%** | Poorly controlled diabetes |
| Blood Pressure | **121/101** | Elevated diastolic |
| SpO2 | **91%** | ⚠️ Hypoxia |
| Medications | Metformin 500mg, Amlodipine 5mg | — |
| Notes | Awaiting specialist report | — |

**Recommended Actions:**
- 🚨 **Immediate outreach** — patient is 340 days overdue with hypoxia (SpO2 91%); arrange urgent in-person or telehealth assessment within 48 hours
- Order **chest X-ray and ABG*

In [ ]:
print("=" * 55)
print("TASK 4 — SINGLE PATIENT ANALYSIS: P0001")
print("=" * 55)

response = run_agent(
    "Analyse patient P0001 in detail. "
    "Retrieve their full record, assess their clinical risk level, "
    "and generate a specific prioritised follow-up action plan. "
    "Include: urgency level (High / Medium / Low), "
    "recommended clinical interventions, and next steps for the care team."
)

print("\n📋 AGENT RESPONSE:")
print(response)

TASK 4 — SINGLE PATIENT ANALYSIS: P0001

📋 AGENT RESPONSE:
## Patient Clinical Analysis: Deepa Krishnan (P0008)

---

### **URGENCY LEVEL: HIGH RISK 🔴**

---

### Patient Summary

| Field | Value |
|-------|-------|
| **Name** | Deepa Krishnan |
| **Age / Gender** | 45 / Male |
| **Primary Diagnosis** | Heart Failure |
| **Current Medication** | Furosemide 40mg (monotherapy) |
| **Days Since Last Visit** | 204 days |
| **Next Scheduled Visit** | July 25, 2024 |

---

### Clinical Risk Findings

#### 🏥 Risk 1: Acute Decompensated Heart Failure
- **BNP:** 1,100.1 pg/mL (critical threshold > 100 pg/mL)
- This value is **11x above the diagnostic threshold** and strongly indicates acute heart failure decompensation.

#### 🩺 Risk 2: Stage 2 Hypertension
- **Blood Pressure:** 155/106 mmHg
- Diastolic pressure ≥100 mmHg meets criteria for a **hypertensive urgency** concern in a heart failure patient.

#### 🫁 Risk 3: Impaired Oxygenation
- **SpO2:** 92% (borderline low; concerning in heart fail

## Task 5 — Missed Appointment Follow-Up

In [19]:
# Step 1: Identify top missed-appointment patients
missed_result = agent_executor.invoke({
    "input": "List the top 5 patients who missed their last appointment, "
             "sorted by how overdue they are."
})
print(missed_result["messages"][-1].content)

---

# 🏥 MediCare Clinical Follow-Up Report

## Population Summary
| Metric | Value |
|---|---|
| Patients reviewed | 10 |
| High lab risk flagged | 5 patients |
| Vitals risk flagged | 6 patients |
| Missed last appointment | 5 patients (top 5 overdue) |

---

## 🔴 Priority 1 — CRITICAL (Immediate Action Within 24-48hrs)

### 1. **P0003 — Rohan Mehta** (45M, Diabetes + CKD)
- **HbA1c: 11.4%** (critical, target <7%), **SpO2: 91%** (hypoxia)
- On Insulin Glargine + Losartan
- **Actions:**
  - 🚨 Urgent clinic assessment — hypoxia + severe hyperglycemia
  - Escalate insulin regimen; consider basal-bolus protocol
  - CKD panel (eGFR, creatinine, electrolytes)
  - Chest X-ray / ABG given low SpO2
  - Diabetic foot review and retinal screening
  - Nephrology liaison for CKD progression

### 2. **P0002 — Priya Patel** (29M, Heart Failure)
- **BNP: 317.6 pg/mL** (>3x threshold >100), **BP: 174/91**
- On Furosemide 40mg
- **Actions:**
  - 🚨 Urgent cardiology review — volume overload likely
  - 

In [21]:
# Step 2: Batch analysis — run the agent for each missed-appointment patient
missed_patients = (
    df[df["missed_last_appointment"].str.lower() == "yes"]
    .sort_values("days_since_last_visit", ascending=False)
    .head(5)["patient_id"]
    .tolist()
)
print(f"Analysing {len(missed_patients)} overdue patients: {missed_patients}\n")

batch_results = []
for pid in missed_patients:
    print(f"--- Analysing {pid} ---")
    r = agent_executor.invoke({
        "input": f"Patient {pid} missed their last appointment and is overdue. "
                 f"Assess all clinical risks and generate a follow-up care action plan."
    })
    batch_results.append({"patient_id": pid, "plan": r["messages"][-1].content})
    print(r["messages"][-1].content)
    print()

Analysing 5 overdue patients: ['P0099', 'P0066', 'P0043', 'P0036', 'P0062']

--- Analysing P0099 ---
All data collected. Here is the complete clinical follow-up analysis.

---

# 🏥 MediCare Clinical Follow-Up Report
**Generated: 2026-06-28 | Population: Top 10 Most Overdue Patients**

---

## 📊 Population Summary

| Metric | Value |
|---|---|
| Patients reviewed | 10 |
| Average days overdue | ~270 |
| Most overdue | 340 days (P0099) |
| High-risk patients | 8 |
| Moderate-risk patients | 2 |
| Top comorbidities | Hypertension (6), Type 2 Diabetes (4), CKD (1), Anemia (1), Asthma (1), Hypothyroidism (1), Osteoarthritis (1) |

---

## 🔴 CRITICAL — Immediate Action Required (Priority 1)

### 1. P0099 — Paramita Chatterjee (70M)
**Diagnoses:** Type 2 Diabetes, Hypertension
**Days Overdue:** 340 | **Missed Appointment:** Yes

| Risk Finding | Value | Threshold |
|---|---|---|
| HbA1c | **9.8%** | >7.0% |
| BP Diastolic | **101 mmHg** | ≥90 |
| SpO2 | **91%** | <95% |

**Clinical Concern:**

In [22]:
# Step 3: Summary dashboard of all batch results
print("="*60)
print("MISSED APPOINTMENT FOLLOW-UP — SUMMARY DASHBOARD")
print("="*60)
for item in batch_results:
    print(f"\n[{item['patient_id']}]")
    # Extract first 3 lines of each plan for the summary view
    print("\n".join(item["plan"].split("\n")[:6]))
    print("...")

MISSED APPOINTMENT FOLLOW-UP — SUMMARY DASHBOARD

[P0099]
All data collected. Here is the complete clinical follow-up analysis.

---

# 🏥 MediCare Clinical Follow-Up Report
**Generated: 2026-06-28 | Population: Top 10 Most Overdue Patients**
...

[P0066]
---

# 🏥 MediCare Clinical Follow-Up Report

## Population Summary
- **15 patients** identified as having missed their last scheduled appointment
...

[P0043]
---

## 🏥 MediCare Clinical Follow-Up Report

### Population Overview
- **10 patients** identified as missed-appointment (overdue 221–340 days)
...

[P0036]
---

# 🏥 MediCare Clinical Follow-Up Report

## Population Summary

...

[P0062]
## 📊 MediCare Clinical Follow-Up Report — 28 June 2026

### Population Overview
- **10 patients** identified as overdue for follow-up (221–340 days since last visit)
- **Common conditions:** Hypertension (6), Type 2 Diabetes (4), CKD (1), Asthma (1), Anemia (1), Hypothyroidism (1), Osteoarthritis (1)
- **Key concern:** 7/10 patients have uncontro